#Install and import necessary libraries

In [ ]:
!pip install -q mp_api pymatgen matminer CBFV

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 107.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━

In [ ]:
import numpy as np
import pandas as pd
from mp_api.client import MPRester
from pymatgen.core import Structure
import matplotlib.pyplot as plt
import warnings

import matminer
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementFraction
from matminer.featurizers.composition import OxidationStates
from matminer.featurizers.composition import ElectronAffinity
warnings.filterwarnings("ignore")
import CBFV

#Data Extraction

In [ ]:
API_KEY = "uODkjuYMFkjLrhJt6XikcocKcLNPSQCT"
mpr=MPRester(API_KEY)

In [ ]:
# Search fields -- "task_ids" replaces "origins" (which doesn't
# track band_gap provenance), and we no longer request "is_metal"
fields = [
    "material_id",
    "formula_pretty",
    "structure",
    "band_gap",
    "composition",
    "task_ids",
]

# Search ABX3-type compounds
docs = mpr.summary.search(
    formula=["ABX3"],
    fields=fields
)

# Filter only true ABX3 compounds
data = []

for doc in docs:

    # Reduced composition
    comp = doc.composition.reduced_composition

    # Stoichiometric amounts
    amounts = sorted(comp.get_el_amt_dict().values())

    # Strict 1:1:3 filter
    if amounts == [1.0, 1.0, 3.0]:

        data.append({
            "material_id": doc.material_id,
            "formula": doc.formula_pretty,

            # Composition
            "composition": comp,

            # Structure
            "structure": doc.structure,

            # Band gap
            "band_gap": doc.band_gap,
        })

# Create dataframe
df = pd.DataFrame(data)

print(f"{len(df)} entries collected")

# ============================================================
# Fetch calc_types (the actual functional/calculation-type info)
# via the separate core materials endpoint, batched, then
# collapse each material's task history down to the single
# functional level of its Static calculation -- which is what
# the reported band_gap actually comes from.
# ============================================================
material_ids = df["material_id"].tolist()

chunk_size = 1000
calc_type_map = {}

for i in range(0, len(material_ids), chunk_size):
    chunk = material_ids[i:i + chunk_size]
    mat_docs = mpr.materials.search(
        material_ids=chunk,
        fields=["material_id", "calc_types"]
    )
    for d in mat_docs:
        calc_type_map[str(d.material_id)] = d.calc_types  # {task_id: CalcType, ...}

print(f"Resolved calc_types for {len(calc_type_map)} materials")


def get_static_functional(material_id):
    d = calc_type_map.get(str(material_id))
    if not d:
        return None

    static_types = {str(v) for v in d.values() if "Static" in str(v)}
    if not static_types:
        return None

    if any("GGA+U" in t for t in static_types):
        return "GGA+U Static"
    elif any("r2SCAN" in t for t in static_types):
        return "r2SCAN Static"
    elif any("PBEsol" in t for t in static_types):
        return "PBEsol Static"
    elif any(t == "GGA Static" for t in static_types):
        return "GGA Static"
    else:
        return " / ".join(sorted(static_types))


df["functional"] = df["material_id"].apply(get_static_functional)

print(df["functional"].value_counts(dropna=False))

# Display dataframe
df.head(5)

Retrieving SummaryDoc documents:   0%|          | 0/4719 [00:00<?, ?it/s]

4719 entries collected


Retrieving 1000 material_ids values in 10 batches:   0%|          | 0/10 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

...

Retrieving MaterialsDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/100 [00:00<?, ?it/s]

Retrieving MaterialsDoc documents:   0%|          | 0/19 [00:00<?, ?it/s]

Resolved calc_types for 4719 materials
functional
GGA Static       3525
GGA+U Static      949
r2SCAN Static     102
None               83
PBEsol Static      60
Name: count, dtype: int64


,material_id,formula,composition,structure,band_gap,functional
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",4.1024,GGA Static
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",0.8071,GGA Static
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",2.0031,GGA+U Static
3,mp-864606,AcCuO3,"(Ac, Cu, O)","[[0. 0. 0.] Ac, [1.9566578 1.9566578 1.9566578...",0.0000,GGA Static
4,mp-861502,AcFeO3,"(Ac, Fe, O)","[[0. 0. 0.] Ac, [1.97678086 1.9767782 1.97678...",1.0084,GGA+U Static


In [ ]:
# Save raw data CSV
df.to_csv("raw_data.csv", index=False)

print("CSV file saved as df7_clean.csv")
print("Final shape:", df.shape)

CSV file saved as df7_clean.csv
Final shape: (4719, 6)


#Data Filtering for true perovskite structures

In [ ]:
from pymatgen.analysis.local_env import CrystalNN
from pymatgen.core import Structure, Element


# ============================================================
# STEP 1: Cation/anion classification via electronegativity
# ============================================================
def classify_sites_by_electronegativity(structure: Structure, en_tolerance=0.3):
    """
    Classifies each site as cation-like or anion-like using Pauling
    electronegativity, avoiding the need for formal oxidation-state
    assignment at this stage. Works uniformly across oxides, halides,
    chalcogenides, and hydrides.

    Returns:
        is_anion: list[bool] aligned with site indices
        anion_elements: set of element symbols classified as anion-like
    """
    elements_in_structure = list({site.specie.symbol for site in structure})
    en_values = {el: Element(el).X for el in elements_in_structure}

    max_en = max(en_values.values())
    anion_elements = {el for el, en in en_values.items()
                       if en >= max_en - en_tolerance}

    is_anion = [site.specie.symbol in anion_elements for site in structure]
    return is_anion, anion_elements


# ============================================================
# STEP 2 + 3: B-site identification (with disambiguation) and
# corner-sharing topology validation
# ============================================================
def check_perovskite_topology(structure: Structure, cnn=None,
                               cn_min=5, cn_max=7, en_tolerance=0.3,
                               bond_length_gap_threshold=0.08):
    """
    Validates perovskite-type corner-sharing octahedral connectivity.

    Steps:
      1. Classify cation/anion sites via electronegativity.
      2. Find candidate B-sites: cations with coordination number
         in [cn_min, cn_max], using (site_index, image) neighbor
         keys to avoid undercounting bonds in primitive cells.
      3. If multiple cation elements qualify as candidate B-sites
         (e.g. a distorted A-site cation whose CN falls in the same
         window), disambiguate using mean anion-bond length -- the
         true B-site cation has the shorter mean bond length. Only
         resolved automatically if the gap between the two shortest
         mean bond lengths exceeds bond_length_gap_threshold;
         otherwise flagged as ambiguous and excluded.
      4. Check pairwise vertex-sharing between confirmed B-site
         octahedra: exactly one shared (site, image) anion vertex
         required (corner-sharing); 0 = non-adjacent (ignored);
         >=2 = edge-/face-sharing (fails validation).
    """
    result = {
        "passes_topology": False,
        "b_site_elements": None,
        "anion_elements": None,
        "n_candidate_b_sites": 0,
        "n_edge_or_face_sharing_pairs": 0,
        "disambiguated_by_bond_length": False,
        "error": None,
        # populated later in Step 4, kept here for column ordering
        "a_site_elements": None,
        "a_site_mean_cn": None,
        "a_site_cn_plausible": None,
    }

    if cnn is None:
        cnn = CrystalNN()

    is_anion, anion_elements = classify_sites_by_electronegativity(
        structure, en_tolerance=en_tolerance)
    result["anion_elements"] = sorted(anion_elements)

    try:
        # --- gather all cation sites with CN in the B-site window ---
        candidates = []  # (site_index, element, {(nbr_idx, image), ...}, mean_dist)
        for i, site in enumerate(structure):
            if is_anion[i]:
                continue
            try:
                nn_info = cnn.get_nn_info(structure, i)
            except Exception:
                continue

            cn = len(nn_info)
            if not (cn_min <= cn <= cn_max):
                continue

            anion_neighbor_keys = set()
            dists = []
            for n in nn_info:
                if not is_anion[n["site_index"]]:
                    continue
                image = tuple(n.get("image", (0, 0, 0)))
                anion_neighbor_keys.add((n["site_index"], image))
                nbr_site = structure[n["site_index"]]
                dist = structure[i].distance(nbr_site, jimage=image)
                dists.append(dist)

            if len(anion_neighbor_keys) < cn_min:
                continue

            mean_dist = float(np.mean(dists)) if dists else np.inf
            candidates.append((i, site.specie.symbol, anion_neighbor_keys, mean_dist))

        result["n_candidate_b_sites"] = len(candidates)
        if len(candidates) < 1:
            result["error"] = "no_valid_cation_octahedral_sites"
            return result, is_anion

        b_elements = sorted({el for _, el, _, _ in candidates})

        # --- disambiguate if multiple cation elements qualify ---
        if len(b_elements) != 1:
            mean_dist_by_element = {}
            for el in b_elements:
                els_dists = [d for _, e, _, d in candidates if e == el]
                mean_dist_by_element[el] = float(np.mean(els_dists))

            sorted_els = sorted(mean_dist_by_element, key=mean_dist_by_element.get)
            shortest, second_shortest = sorted_els[0], sorted_els[1]
            gap = (mean_dist_by_element[second_shortest] - mean_dist_by_element[shortest]) \
                  / mean_dist_by_element[shortest]

            if gap < bond_length_gap_threshold:
                result["error"] = f"ambiguous_B_elements_close_bond_lengths: {mean_dist_by_element}"
                return result, is_anion

            true_b_element = shortest
            candidates = [c for c in candidates if c[1] == true_b_element]
            b_elements = [true_b_element]
            result["disambiguated_by_bond_length"] = True

        result["b_site_elements"] = b_elements

        # --- corner-sharing check among confirmed B-site octahedra ---
        bad_pairs = 0
        for idx_a in range(len(candidates)):
            i, el_i, keys_i, _ = candidates[idx_a]
            for idx_b in range(idx_a + 1, len(candidates)):
                j, el_j, keys_j, _ = candidates[idx_b]
                shared = keys_i & keys_j
                if len(shared) == 0:
                    continue
                if len(shared) != 1:
                    bad_pairs += 1

        result["n_edge_or_face_sharing_pairs"] = bad_pairs
        if bad_pairs == 0:
            result["passes_topology"] = True
        else:
            result["error"] = "edge_or_face_sharing_detected"

    except Exception as e:
        result["error"] = str(e)

    return result, is_anion


# ============================================================
# STEP 4: A-site identification
# ============================================================
def identify_a_site(structure, is_anion, b_site_elements, cnn=None,
                     a_cn_min=6, a_cn_max=14):
    """
    Identifies the A-site cation element(s): whatever cation
    element(s) remain in the formula after removing anions and the
    confirmed B-site element(s). Also computes the mean coordination
    number for a plausibility sanity check (typical A-site CN is
    roughly 6-12, sometimes higher in the ideal cubic case, though
    tilting/distortion can depress this).
    """
    if cnn is None:
        cnn = CrystalNN()

    if not b_site_elements:
        return None, None, None

    all_elements = {site.specie.symbol for site in structure}
    anion_els = {structure[i].specie.symbol for i in range(len(structure)) if is_anion[i]}
    candidate_a_elements = all_elements - anion_els - set(b_site_elements)

    if not candidate_a_elements:
        return None, None, None

    site_cns = {el: [] for el in candidate_a_elements}
    for i, site in enumerate(structure):
        el = site.specie.symbol
        if el not in candidate_a_elements:
            continue
        try:
            nn_info = cnn.get_nn_info(structure, i)
        except Exception:
            continue
        site_cns[el].append(len(nn_info))

    a_site_elements = sorted(candidate_a_elements)
    mean_cn_by_element = {
        el: (float(np.mean(cns)) if cns else None) for el, cns in site_cns.items()
    }

    a_site_cn_ok = all(
        (v is not None and a_cn_min <= v <= a_cn_max) for v in mean_cn_by_element.values()
    )

    return a_site_elements, mean_cn_by_element, a_site_cn_ok


# ============================================================
# MAIN: apply the full pipeline to every row of df
# ============================================================
def process_row(structure, cnn=None, cn_min=5, cn_max=7,
                 en_tolerance=0.3, bond_length_gap_threshold=0.08,
                 a_cn_min=6, a_cn_max=14):
    """
    Runs classification -> B-site identification -> topology check
    -> A-site identification, returning one combined result dict.
    """
    if cnn is None:
        cnn = CrystalNN()

    result, is_anion = check_perovskite_topology(
        structure, cnn=cnn, cn_min=cn_min, cn_max=cn_max,
        en_tolerance=en_tolerance,
        bond_length_gap_threshold=bond_length_gap_threshold
    )

    # Only attempt A-site identification if a B-site was confirmed
    # (even if the structure ultimately fails the corner-sharing
    # check, having the A-site on record can be useful diagnostically)
    if result["b_site_elements"]:
        a_elements, a_cn_by_el, a_cn_ok = identify_a_site(
            structure, is_anion, result["b_site_elements"], cnn=cnn,
            a_cn_min=a_cn_min, a_cn_max=a_cn_max
        )
        result["a_site_elements"] = a_elements
        result["a_site_mean_cn"] = a_cn_by_el
        result["a_site_cn_plausible"] = a_cn_ok

    return result


cnn = CrystalNN()  # instantiate once, reuse across all rows (avoids repeated setup cost)

topology_results = []
for idx, row in df.iterrows():
    res = process_row(row["structure"], cnn=cnn)
    res["material_id"] = row["material_id"]
    topology_results.append(res)

topology_df = pd.DataFrame(topology_results)
df_with_topology = df.merge(topology_df, on="material_id", how="left")

# ---------------------------------------------------------
# df1: structures that pass the corner-sharing octahedral
# topology check -- validated dataset with A-site, B-site, and
# anion (X-site) elements all tagged
# ---------------------------------------------------------
df1 = df_with_topology[df_with_topology["passes_topology"] == True].reset_index(drop=True)

# Convenience: single-element helper columns (most rows will have
# exactly one A element, one B element, and one X element; these
# unpack the lists for easier downstream use, e.g. in Shannon
# radius lookups)
df1["a_site_element"] = df1["a_site_elements"].apply(
    lambda els: els[0] if els and len(els) == 1 else els)
df1["b_site_element"] = df1["b_site_elements"].apply(
    lambda els: els[0] if els and len(els) == 1 else els)
df1["x_site_element"] = df1["anion_elements"].apply(
    lambda els: els[0] if els and len(els) == 1 else els)

print(f"Initial entries          : {len(df)}")
print(f"Passed topology check    : {len(df1)}")
print(f"Failed                   : {len(df) - len(df1)}")
print(df_with_topology["error"].value_counts(dropna=False))

# Sanity check: how many validated entries have a single, unambiguous
# A-site / X-site element vs. multiple (mixed-site, e.g. solid solutions)?
print(df1["a_site_elements"].apply(lambda x: len(x) if x else 0).value_counts())
print(df1["anion_elements"].apply(lambda x: len(x) if x else 0).value_counts())

df1

Initial entries          : 4719
Passed topology check    : 2592
Failed                   : 2127
error
None                                                                                            2592
no_valid_cation_octahedral_sites                                                                1220
edge_or_face_sharing_detected                                                                    616
ambiguous_B_elements_close_bond_lengths: {'Mg': 2.6770795034585118, 'Nd': 2.857566659657669}       1
ambiguous_B_elements_close_bond_lengths: {'Lu': 2.7586635557709864, 'Nd': 2.809287850318846}       1
                                                                                                ... 
ambiguous_B_elements_close_bond_lengths: {'Bi': 2.434743906275288, 'La': 2.454289863656188}        1
ambiguous_B_elements_close_bond_lengths: {'Ba': 2.7591709005875273, 'K': 2.9740857473893794}       1
ambiguous_B_elements_close_bond_lengths: {'In': 2.817384886777946, 'Sb': 2.912474737708682

,material_id,formula,composition,structure,band_gap,functional,passes_topology,b_site_elements,anion_elements,n_candidate_b_sites,n_edge_or_face_sharing_pairs,disambiguated_by_bond_length,error,a_site_elements,a_site_mean_cn,a_site_cn_plausible,a_site_element,b_site_element,x_site_element
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",4.1024,GGA Static,True,[Al],[O],1,0,False,None,[Ac],{'Ac': 12.0},True,Ac,Al,O
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",0.8071,GGA Static,True,[B],[O],1,0,False,None,[Ac],{'Ac': 12.0},True,Ac,B,O
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",2.0031,GGA+U Static,True,[Cr],[O],1,0,False,None,[Ac],{'Ac': 12.0},True,Ac,Cr,O
3,mp-864606,AcCuO3,"(Ac, Cu, O)","[[0. 0. 0.] Ac, [1.9566578 1.9566578 1.9566578...",0.0000,GGA Static,True,[Cu],[O],1,0,False,None,[Ac],{'Ac': 12.0},True,Ac,Cu,O
4,mp-861502,AcFeO3,"(Ac, Fe, O)","[[0. 0. 0.] Ac, [1.97678086 1.9767782 1.97678...",1.0084,GGA+U Static,True,[Fe],[O],1,0,False,None,[Ac],{'Ac': 12.0},True,Ac,Fe,O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2587,mp-1183043,ZrSiO3,"(Zr, Si, O)","[[0. 0. 0.] Zr, [1.8317365 1.8317365 1.8317365...",0.0000,GGA Static,True,[Si],[O],1,0,False,None,[Zr],{'Zr': 12.0},True,Zr,Si,O
2588,mp-1183045,ZrTiO3,"(Zr, Ti, O)","[[0. 0. 0.] Zr, [1.9295495 1.9295495 1.9295495...",0.0000,GGA Static,True,[Ti],[O],1,0,False,None,[Zr],{'Zr': 12.0},True,Zr,Ti,O
2589,mp-1183044,ZrTlO3,"(Zr, Tl, O)","[[2.106323 2.106323 2.106323] Zr, [0. 0. 0.] T...",0.0000,GGA Static,True,[Zr],[O],1,0,False,None,[Tl],{'Tl': 0.0},False,Tl,Zr,O
2590,mp-1246572,ZrWN3,"(Zr, W, N)","[[-0.14222385 2.89500372 2.97756927] Zr, [3....",0.0000,GGA Static,True,[W],[N],8,0,True,None,[Zr],{'Zr': 5.5},False,Zr,W,N


#Radius determination

In [ ]:
from pymatgen.core import Composition

KNOWN_ANION_OXI_STATES = {
    "O": -2, "S": -2, "Se": -2, "Te": -2,
    "F": -1, "Cl": -1, "Br": -1, "I": -1,
    "N": -3, "H": -1,
}

def assign_oxidation_states_by_site(row):
    result = {"a_oxi_state": None, "b_oxi_state": None, "x_oxi_state": None,
               "oxi_error": None}   # <-- renamed from "error"

    a_el = row["a_site_element"]
    b_el = row["b_site_element"]
    x_el = row["x_site_element"]
    comp = row["composition"]

    if not isinstance(a_el, str) or not isinstance(b_el, str) or not isinstance(x_el, str):
        result["oxi_error"] = "non_single_element_site_assignment"
        return result

    if x_el not in KNOWN_ANION_OXI_STATES:
        result["oxi_error"] = f"unknown_anion_reference_oxidation_state: {x_el}"
        return result

    expected_x_oxi = KNOWN_ANION_OXI_STATES[x_el]

    try:
        guesses = comp.oxi_state_guesses(max_sites=-1)
    except Exception as e:
        result["oxi_error"] = f"oxi_state_guesses_failed: {e}"
        return result

    if not guesses:
        result["oxi_error"] = "no_valid_oxi_state_combination_found"
        return result

    for guess in guesses:
        if guess.get(x_el) == expected_x_oxi:
            if a_el in guess and b_el in guess:
                result["a_oxi_state"] = guess[a_el]
                result["b_oxi_state"] = guess[b_el]
                result["x_oxi_state"] = expected_x_oxi
                return result

    result["oxi_error"] = "no_guess_matched_expected_X_oxidation_state"
    return result


oxi_results = df1.apply(assign_oxidation_states_by_site, axis=1, result_type="expand")
df2 = pd.concat([df1, oxi_results], axis=1)

print(df2["oxi_error"].value_counts(dropna=False))
df2.head(2)

oxi_error
NaN                                            2079
no_valid_oxi_state_combination_found            446
non_single_element_site_assignment               34
no_guess_matched_expected_X_oxidation_state      19
unknown_anion_reference_oxidation_state: Au       4
unknown_anion_reference_oxidation_state: Pt       4
unknown_anion_reference_oxidation_state: C        4
unknown_anion_reference_oxidation_state: Sb       1
unknown_anion_reference_oxidation_state: Ge       1
Name: count, dtype: int64


,material_id,formula,composition,structure,band_gap,functional,passes_topology,b_site_elements,anion_elements,n_candidate_b_sites,...,a_site_elements,a_site_mean_cn,a_site_cn_plausible,a_site_element,b_site_element,x_site_element,a_oxi_state,b_oxi_state,x_oxi_state,oxi_error
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",4.1024,GGA Static,True,[Al],[O],1,...,[Ac],{'Ac': 12.0},True,Ac,Al,O,3.0,3.0,-2.0,NaN
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",0.8071,GGA Static,True,[B],[O],1,...,[Ac],{'Ac': 12.0},True,Ac,B,O,3.0,3.0,-2.0,NaN


In [ ]:
def is_clean_string(val):
    return isinstance(val, str)

for site in ["a", "b", "x"]:
    col = f"{site}_site_element"
    n_list_valued = (~df2[col].apply(is_clean_string)).sum()
    print(f"{site}-site: {n_list_valued} rows have a non-single-element (list) assignment")

# these are mixed-site compositions (e.g. solid solutions) that
# don't have a single well-defined radius, and should be handled
# separately (flagged/excluded) rather than crashing the lookup.
for site in ["a", "b", "x"]:
    col = f"{site}_site_element"
    df2[col] = df2[col].apply(lambda v: v if isinstance(v, str) else np.nan)

a-site: 34 rows have a non-single-element (list) assignment
b-site: 0 rows have a non-single-element (list) assignment
x-site: 34 rows have a non-single-element (list) assignment


In [ ]:
from pymatgen.analysis.local_env import CrystalNN
import numpy as np

cnn = CrystalNN()

def get_mean_cn_for_element(structure, target_element, cnn):
    if not isinstance(target_element, str):
        return np.nan
    cns = []
    for i, site in enumerate(structure):
        if site.specie.symbol != target_element:
            continue
        try:
            nn_info = cnn.get_nn_info(structure, i)
            cns.append(len(nn_info))
        except Exception:
            continue
    return float(np.mean(cns)) if cns else np.nan

df2["b_site_cn"] = df2.apply(
    lambda row: get_mean_cn_for_element(row["structure"], row["b_site_element"], cnn),
    axis=1
)

df2["x_site_cn"] = df2.apply(
    lambda row: get_mean_cn_for_element(row["structure"], row["x_site_element"], cnn),
    axis=1
)

# a_site_cn: recompute directly rather than relying on the
# a_site_mean_cn dict (which may also have been lost/misaligned
# in the same way) -- this is simpler and more robust
df2["a_site_cn"] = df2.apply(
    lambda row: get_mean_cn_for_element(row["structure"], row["a_site_element"], cnn),
    axis=1
)

print(df2[["a_site_cn", "b_site_cn", "x_site_cn"]].isna().sum())

a_site_cn    34
b_site_cn     0
x_site_cn    34
dtype: int64


In [ ]:
from pymatgen.core import Species, Element

# ============================================================
# Level definitions (most physically grounded -> least):
#   1. Shannon radius at the EXACT structurally-measured CN
#   2. Shannon radius at the nearest tabulated CN (already
#      built into your fallback list)
#   3. Species/Element average ionic radius (CN-independent
#      Shannon-derived average) -- still uses the correct
#      oxidation state, just not CN-specific
#   4. Atomic radius (oxidation-state-independent, last resort
#      when ionic radius truly isn't tabulated)
#   5. If oxidation state itself failed: assume the element's
#      most common/typical oxidation state and retry 1-4
#   6. Data-driven: median radius for the SAME element+site-role
#      already computed elsewhere in your dataset
#   7. Final fallback: median radius for that SITE ROLE (A/B/X)
#      across the whole dataset
# ============================================================

CN_TO_ROMAN = {
    2: "II", 3: "III", 4: "IV", 5: "V", 6: "VI", 7: "VII",
    8: "VIII", 9: "IX", 10: "X", 11: "XI", 12: "XII"
}

def cn_to_roman(cn):
    if cn is None or (isinstance(cn, float) and np.isnan(cn)):
        return None
    cn_int = int(round(cn))
    nearest = min(CN_TO_ROMAN.keys(), key=lambda x: abs(x - cn_int))
    return CN_TO_ROMAN[nearest]


def try_shannon_exact(sp, cn):
    roman_cn = cn_to_roman(cn)
    if roman_cn is None:
        return None
    try:
        r = sp.get_shannon_radius(cn=roman_cn)
        return float(r) if r is not None else None
    except Exception:
        return None


def try_shannon_nearby(sp):
    for fallback_cn in ["VI", "VIII", "XII", "IV", "III", "II"]:
        try:
            r = sp.get_shannon_radius(cn=fallback_cn)
            if r is not None:
                return float(r)
        except Exception:
            continue
    return None


def try_average_ionic_radius(element_symbol, oxi_state):
    try:
        sp = Species(element_symbol, oxi_state)
        r = sp.ionic_radius  # CN-independent average, still oxidation-state-aware
        return float(r) if r is not None else None
    except Exception:
        return None


def try_atomic_radius(element_symbol):
    try:
        el = Element(element_symbol)
        return float(el.atomic_radius) if el.atomic_radius is not None else None
    except Exception:
        return None


def try_typical_oxidation_state(element_symbol):
    """Fall back to the element's most common tabulated oxidation
    state when the structural/charge-balance assignment failed."""
    try:
        el = Element(element_symbol)
        common = el.common_oxidation_states
        return common[0] if common else None
    except Exception:
        return None


def get_radius_with_fallback(element_symbol, oxi_state, cn):
    """
    Returns (radius, method_used) so every value can be audited.
    """
    if element_symbol is None or (isinstance(element_symbol, float) and np.isnan(element_symbol)):
        return np.nan, "no_element"

    oxi_missing = oxi_state is None or (isinstance(oxi_state, float) and np.isnan(oxi_state))
    used_typical_oxi = False

    if oxi_missing:
        oxi_state = try_typical_oxidation_state(element_symbol)
        used_typical_oxi = oxi_state is not None

    if oxi_state is not None:
        try:
            sp = Species(element_symbol, oxi_state)
        except Exception:
            sp = None

        if sp is not None:
            r = try_shannon_exact(sp, cn)
            if r is not None:
                return r, ("typical_oxi_shannon_exact_cn" if used_typical_oxi else "shannon_exact_cn")

            r = try_shannon_nearby(sp)
            if r is not None:
                return r, ("typical_oxi_shannon_fallback_cn" if used_typical_oxi else "shannon_fallback_cn")

            r = try_average_ionic_radius(element_symbol, oxi_state)
            if r is not None:
                return r, ("typical_oxi_average_ionic_radius" if used_typical_oxi else "average_ionic_radius")

    r = try_atomic_radius(element_symbol)
    if r is not None:
        return r, "atomic_radius"

    return np.nan, "unresolved"


# ============================================================
# Apply to each site, tracking radius + method used
# ============================================================
for site in ["a", "b", "x"]:
    df2[f"{site}_site_radius"], df2[f"{site}_site_radius_method"] = zip(*df2.apply(
        lambda row: get_radius_with_fallback(
            row[f"{site}_site_element"], row[f"{site}_oxi_state"], row[f"{site}_site_cn"]
        ),
        axis=1
    ))

# ============================================================
# Levels 6-7: data-driven fallback for anything STILL unresolved
# after the physically-grounded chain above
# ============================================================
for site in ["a", "b", "x"]:
    radius_col = f"{site}_site_radius"
    method_col = f"{site}_site_radius_method"
    element_col = f"{site}_site_element"

    still_missing = df2[radius_col].isna()

    if still_missing.any():
        # Level 6: median radius for the SAME element, computed
        # from other rows in the dataset where it succeeded
        element_medians = df2.loc[~still_missing].groupby(element_col)[radius_col].median()

        for idx in df2[still_missing].index:
            el = df2.loc[idx, element_col]
            if el in element_medians.index and not np.isnan(element_medians[el]):
                df2.loc[idx, radius_col] = element_medians[el]
                df2.loc[idx, method_col] = "dataset_median_same_element"

    # Level 7: final fallback -- median radius for this SITE ROLE
    # across the whole dataset (only hit if the same element never
    # succeeds anywhere else in the dataset either)
    still_missing = df2[radius_col].isna()
    if still_missing.any():
        role_median = df2[radius_col].median()
        df2.loc[still_missing, radius_col] = role_median
        df2.loc[still_missing, method_col] = "dataset_median_site_role"

df2["a_site_radius"] = df2["a_site_radius"].abs()
df2["x_site_radius"] = df2["x_site_radius"].abs()

# ============================================================
# Audit summary -- report this in your methods/response text
# ============================================================
for site in ["a", "b", "x"]:
    print(f"\n--- {site.upper()}-site radius source breakdown ---")
    print(df2[f"{site}_site_radius_method"].value_counts(dropna=False))
    print(f"Remaining NaN: {df2[f'{site}_site_radius'].isna().sum()}")


--- A-site radius source breakdown ---
a_site_radius_method
shannon_exact_cn                   1395
shannon_fallback_cn                 584
typical_oxi_shannon_exact_cn        325
typical_oxi_shannon_fallback_cn     137
atomic_radius                       101
dataset_median_site_role             34
average_ionic_radius                 16
Name: count, dtype: int64
Remaining NaN: 0

--- B-site radius source breakdown ---
b_site_radius_method
shannon_exact_cn                   1617
typical_oxi_shannon_exact_cn        429
shannon_fallback_cn                 207
average_ionic_radius                154
atomic_radius                       143
typical_oxi_shannon_fallback_cn      42
Name: count, dtype: int64
Remaining NaN: 0

--- X-site radius source breakdown ---
x_site_radius_method
shannon_exact_cn                   1401
shannon_fallback_cn                 660
typical_oxi_shannon_exact_cn        396
typical_oxi_shannon_fallback_cn      74
dataset_median_site_role             34
atomic_radi

In [ ]:
df2.head(3)

,material_id,formula,composition,structure,band_gap,functional,passes_topology,b_site_elements,anion_elements,n_candidate_b_sites,...,oxi_error,b_site_cn,x_site_cn,a_site_cn,a_site_radius,a_site_radius_method,b_site_radius,b_site_radius_method,x_site_radius,x_site_radius_method
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",4.1024,GGA Static,True,[Al],[O],1,...,NaN,6.0,2.0,12.0,1.12,shannon_fallback_cn,0.535,shannon_exact_cn,1.35,shannon_exact_cn
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",0.8071,GGA Static,True,[B],[O],1,...,NaN,6.0,6.0,12.0,1.12,shannon_fallback_cn,0.270,shannon_exact_cn,1.40,shannon_exact_cn
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",2.0031,GGA+U Static,True,[Cr],[O],1,...,NaN,6.0,2.0,12.0,1.12,shannon_fallback_cn,0.615,shannon_exact_cn,1.35,shannon_exact_cn


In [ ]:
# Save radius_calculation data CSV
df2.to_csv("radius_calculation.csv", index=False)

print("CSV file saved as radius_calculation.csv")
print("Final shape:", df2.shape)

CSV file saved as radius_calculation.csv
Final shape: (2592, 32)


#Feature Generations

In [ ]:
# Create a smaller dataframe with selected columns only

df3 = df2[["material_id",
    "formula",
    "functional",
    "composition",
    "structure",
    "band_gap",
            "a_site_element",
            "b_site_element",
            "x_site_element",
            "a_site_radius",
            "b_site_radius",
            "x_site_radius",
]].copy()

# Display dataframe
df3

,material_id,formula,functional,composition,structure,band_gap,a_site_element,b_site_element,x_site_element,a_site_radius,b_site_radius,x_site_radius
0,mp-1183115,AcAlO3,GGA Static,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",4.1024,Ac,Al,O,1.120,0.535,1.35
1,mp-1183052,AcBO3,GGA Static,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",0.8071,Ac,B,O,1.120,0.270,1.40
2,mp-866101,AcCrO3,GGA+U Static,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",2.0031,Ac,Cr,O,1.120,0.615,1.35
3,mp-864606,AcCuO3,GGA Static,"(Ac, Cu, O)","[[0. 0. 0.] Ac, [1.9566578 1.9566578 1.9566578...",0.0000,Ac,Cu,O,1.120,0.540,1.40
4,mp-861502,AcFeO3,GGA+U Static,"(Ac, Fe, O)","[[0. 0. 0.] Ac, [1.97678086 1.9767782 1.97678...",1.0084,Ac,Fe,O,1.120,0.780,1.35
...,...,...,...,...,...,...,...,...,...,...,...,...
2587,mp-1183043,ZrSiO3,GGA Static,"(Zr, Si, O)","[[0. 0. 0.] Zr, [1.8317365 1.8317365 1.8317365...",0.0000,Zr,Si,O,1.550,0.400,1.35
2588,mp-1183045,ZrTiO3,GGA Static,"(Zr, Ti, O)","[[0. 0. 0.] Zr, [1.9295495 1.9295495 1.9295495...",0.0000,Zr,Ti,O,1.550,0.605,1.35
2589,mp-1183044,ZrTlO3,GGA Static,"(Zr, Tl, O)","[[2.106323 2.106323 2.106323] Zr, [0. 0. 0.] T...",0.0000,Tl,Zr,O,0.885,1.550,1.35
2590,mp-1246572,ZrWN3,GGA Static,"(Zr, W, N)","[[-0.14222385 2.89500372 2.97756927] Zr, [3....",0.0000,Zr,W,N,1.550,0.510,1.46


In [ ]:
# Create a smaller dataframe with selected columns only

df3n = df3[["material_id",
           #"pretty_formula",
    "formula",
    "composition",
    "structure",
    "functional",
    "band_gap",
            "a_site_radius",
            "b_site_radius",
            "x_site_radius",
]].copy()

# Display dataframe
df3n.head()

,material_id,formula,composition,structure,functional,band_gap,a_site_radius,b_site_radius,x_site_radius
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,1.12,0.535,1.35
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,1.12,0.270,1.40
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",GGA+U Static,2.0031,1.12,0.615,1.35
3,mp-864606,AcCuO3,"(Ac, Cu, O)","[[0. 0. 0.] Ac, [1.9566578 1.9566578 1.9566578...",GGA Static,0.0000,1.12,0.540,1.40
4,mp-861502,AcFeO3,"(Ac, Fe, O)","[[0. 0. 0.] Ac, [1.97678086 1.9767782 1.97678...",GGA+U Static,1.0084,1.12,0.780,1.35


##Elemental Feature Generation using CBFV

In [ ]:
from CBFV import composition
import pandas as pd

# Prepare dataframe for CBFV
# CBFV expects columns named exactly: formula and target/property

cbfv_input = df3n[["formula", "band_gap"]].copy()
cbfv_input.columns = ["formula", "target"]

# Generate Oliynyk features
X, y, formulae, skipped = composition.generate_features(
    cbfv_input,
    elem_prop='oliynyk'
)

# Create dataframe of generated features
features_df = pd.DataFrame(X)

# Add original columns back
df4 = df3n.merge(features_df, left_index=True, right_index=True)

# Display final dataframe
df4.head(3)

Processing Input Data: 100%|██████████| 2592/2592 [00:00<00:00, 22520.22it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 2592/2592 [00:00<00:00, 11207.04it/s]



NOTE: Your data contains formula with exotic elements. These were skipped.
	Creating Pandas Objects...


,material_id,formula,composition,structure,functional,band_gap,a_site_radius,b_site_radius,x_site_radius,avg_Atomic_Number,...,mode_polarizability(A^3),mode_Melting_point_(K),mode_Boiling_Point_(K),mode_Density_(g/mL),mode_specific_heat_(J/g_K)_,mode_heat_of_fusion_(kJ/mol)_,mode_heat_of_vaporization_(kJ/mol)_,mode_thermal_conductivity_(W/(m_K))_,mode_heat_atomization(kJ/mol),mode_Cohesive_energy
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,1.12,0.535,1.35,7.4,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,1.12,0.270,1.40,5.8,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",GGA+U Static,2.0031,1.12,0.615,1.35,9.6,...,0.793,54.75,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62


##Elemental Feature Generation from matminer

In [ ]:
from matminer.featurizers.conversions import CompositionToOxidComposition

# ---------------------------------------------------
# Create oxidation-state compositions
# ---------------------------------------------------

oxid_featurizer = CompositionToOxidComposition()
df5 = df4.copy()

df5 = oxid_featurizer.featurize_dataframe(
    df5,
    col_id="composition",
    ignore_errors=True
)

# Generated column name
oxid_col = "composition_oxid"

# ---------------------------------------------------
# Add oxidation states to structures
# ---------------------------------------------------

decorated_structures = []

for comp_oxi, structure in zip(df5[oxid_col], df5["structure"]):

    try:
        # Create oxidation-state dictionary
        oxi_dict = {}

        for sp in comp_oxi.elements:
            oxi_dict[sp.element.symbol] = sp.oxi_state

        # Copy structure
        s = structure.copy()

        # Decorate structure
        s.add_oxidation_state_by_element(oxi_dict)

        decorated_structures.append(s)

    except:
        decorated_structures.append(np.nan)

# Add oxidized structures column
df5["structure_oxi"] = decorated_structures

df5.head(2)

CompositionToOxidComposition:   0%|          | 0/2592 [00:00<?, ?it/s]

,material_id,formula,composition,structure,functional,band_gap,a_site_radius,b_site_radius,x_site_radius,avg_Atomic_Number,...,mode_Boiling_Point_(K),mode_Density_(g/mL),mode_specific_heat_(J/g_K)_,mode_heat_of_fusion_(kJ/mol)_,mode_heat_of_vaporization_(kJ/mol)_,mode_thermal_conductivity_(W/(m_K))_,mode_heat_atomization(kJ/mol),mode_Cohesive_energy,composition_oxid,structure_oxi
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,1.12,0.535,1.35,7.4,...,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62,"(Ac3+, Al3+, O2-)","[[0. 0. 0.] Ac3+, [1.92931693 1.92931693 1.929..."
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,1.12,0.270,1.40,5.8,...,90.15,0.00143,0.92,0.22259,3.4099,0.02674,249.0,2.62,"(Ac3+, B3+, O2-)","[[0. 0. 0.] Ac3+, [1.860834 1.860834 1.860834]..."


In [ ]:
from matminer.featurizers.composition import (
    IonProperty,
    BandCenter,
)

from matminer.featurizers.base import MultipleFeaturizer
from pymatgen.core import Composition
import pandas as pd

# Make a copy
#df5 = df4_lowcorr.copy()

# Ensure composition column contains pymatgen Composition objects
df5["composition"] = df5["composition"].apply(
    lambda x: x if isinstance(x, Composition) else Composition(str(x))
)

# Initialize selected composition featurizers
featurizer = MultipleFeaturizer([
    IonProperty(),
    BandCenter()
])

# Generate features directly from existing composition column
df5 = featurizer.featurize_dataframe(
    df5,
    col_id="composition",
    ignore_errors=True
)

# Display newly added feature names
new_features = featurizer.feature_labels()

print("Added matminer features:")
print(new_features)

# Display final dataframe
df5.head(2)

MultipleFeaturizer:   0%|          | 0/2592 [00:00<?, ?it/s]

Added matminer features:
['compound possible', 'max ionic char', 'avg ionic char', 'band center']


,material_id,formula,composition,structure,functional,band_gap,a_site_radius,b_site_radius,x_site_radius,avg_Atomic_Number,...,mode_heat_of_vaporization_(kJ/mol)_,mode_thermal_conductivity_(W/(m_K))_,mode_heat_atomization(kJ/mol),mode_Cohesive_energy,composition_oxid,structure_oxi,compound possible,max ionic char,avg ionic char,band center
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,1.12,0.535,1.35,7.4,...,3.4099,0.02674,249.0,2.62,"(Ac3+, Al3+, O2-)","[[0. 0. 0.] Ac3+, [1.92931693 1.92931693 1.929...",True,0.745613,0.160043,5.743996
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,1.12,0.270,1.40,5.8,...,3.4099,0.02674,249.0,2.62,"(Ac3+, B3+, O2-)","[[0. 0. 0.] Ac3+, [1.860834 1.860834 1.860834]...",True,0.745613,0.143887,6.085109


In [ ]:
from matminer.featurizers.composition import AtomicOrbitals, ValenceOrbital

df5 = AtomicOrbitals().featurize_dataframe(df5, "composition", ignore_errors=True)
df5 = ValenceOrbital().featurize_dataframe(df5, "composition", ignore_errors=True)

df5.head(2)

AtomicOrbitals:   0%|          | 0/2592 [00:00<?, ?it/s]

ValenceOrbital:   0%|          | 0/2592 [00:00<?, ?it/s]

,material_id,formula,composition,structure,functional,band_gap,a_site_radius,b_site_radius,x_site_radius,avg_Atomic_Number,...,LUMO_energy,gap_AO,avg s valence electrons,avg p valence electrons,avg d valence electrons,avg f valence electrons,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,1.12,0.535,1.35,7.4,...,-0.286883,0.051498,2.0,2.6,0.2,0.0,0.416667,0.541667,0.041667,0.0
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,1.12,0.270,1.40,5.8,...,-0.338381,0.000000,2.0,2.6,0.2,0.0,0.416667,0.541667,0.041667,0.0


##Geometrical Feature Generation

In [ ]:
#moving radius columns to last to create different feature set
cols_to_move = ["a_site_radius", "b_site_radius", "x_site_radius"]

df5 = df5[
    [col for col in df5.columns if col not in cols_to_move] + cols_to_move
]
df5.head(2)

,material_id,formula,composition,structure,functional,band_gap,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,...,avg p valence electrons,avg d valence electrons,avg f valence electrons,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,a_site_radius,b_site_radius,x_site_radius
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,7.4,14.995948,1.8,12.2,...,2.6,0.2,0.0,0.416667,0.541667,0.041667,0.0,1.12,0.535,1.35
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,5.8,11.761840,1.6,12.2,...,2.6,0.2,0.0,0.416667,0.541667,0.041667,0.0,1.12,0.270,1.40


In [ ]:
# Tolerance factor
df5['Tolerance_factor'] = (
    (df5['a_site_radius'] + df5['x_site_radius']) /
    (
        np.sqrt(2) *
        (df5['b_site_radius'] + df5['x_site_radius'])
    )
)

# Octahedral factor
df5['Octahedral_factor'] = (
    df5['b_site_radius'] /
    df5['x_site_radius']
)

df5.head(2)

,material_id,formula,composition,structure,functional,band_gap,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,...,avg f valence electrons,frac s valence electrons,frac p valence electrons,frac d valence electrons,frac f valence electrons,a_site_radius,b_site_radius,x_site_radius,Tolerance_factor,Octahedral_factor
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,7.4,14.995948,1.8,12.2,...,0.0,0.416667,0.541667,0.041667,0.0,1.12,0.535,1.35,0.926554,0.396296
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,5.8,11.761840,1.6,12.2,...,0.0,0.416667,0.541667,0.041667,0.0,1.12,0.270,1.40,1.067011,0.192857


##Structural Feature Generation

In [ ]:
from matminer.featurizers.structure import GlobalSymmetryFeatures
from matminer.featurizers.structure import MinimumRelativeDistances, EwaldEnergy

gsf = GlobalSymmetryFeatures()

df6 = gsf.featurize_dataframe(df5, col_id="structure")
df6.head(3)

GlobalSymmetryFeatures:   0%|          | 0/2592 [00:00<?, ?it/s]

,material_id,formula,composition,structure,functional,band_gap,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,...,a_site_radius,b_site_radius,x_site_radius,Tolerance_factor,Octahedral_factor,spacegroup_num,crystal_system,crystal_system_int,is_centrosymmetric,n_symmetry_ops
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,7.4,14.995948,1.8,12.2,...,1.12,0.535,1.35,0.926554,0.396296,221,cubic,1,True,48
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,5.8,11.761840,1.6,12.2,...,1.12,0.270,1.40,1.067011,0.192857,221,cubic,1,True,96
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",GGA+U Static,2.0031,9.6,19.998860,2.0,10.8,...,1.12,0.615,1.35,0.888831,0.455556,221,cubic,1,True,48


In [ ]:
from matminer.featurizers.structure import (
    DensityFeatures,
    MaximumPackingEfficiency,
)

featurizers = [
    DensityFeatures(),
    MaximumPackingEfficiency(),
]

for f in featurizers:
    df6 = f.featurize_dataframe(df=df6, col_id="structure")

df6.head(3)

DensityFeatures:   0%|          | 0/2592 [00:00<?, ?it/s]

MaximumPackingEfficiency:   0%|          | 0/2592 [00:00<?, ?it/s]

,material_id,formula,composition,structure,functional,band_gap,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,...,Octahedral_factor,spacegroup_num,crystal_system,crystal_system_int,is_centrosymmetric,n_symmetry_ops,density,vpa,packing fraction,max packing efficiency
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,7.4,14.995948,1.8,12.2,...,0.396296,221,cubic,1,True,48,8.728230,11.490283,0.730268,0.446920
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,5.8,11.761840,1.6,12.2,...,0.192857,221,cubic,1,True,96,9.206879,10.309625,0.705091,0.446920
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",GGA+U Static,2.0031,9.6,19.998860,2.0,10.8,...,0.455556,221,cubic,1,True,48,8.848788,12.272569,0.737706,0.446918


In [ ]:
from matminer.featurizers.structure import GlobalInstabilityIndex

# Calculate GII
# ---------------------------------------------------
df7=df6.copy()
gii_featurizer = GlobalInstabilityIndex()

gii_values = []

for s in df7["structure_oxi"]:

    try:
        gii = gii_featurizer.featurize(s)[0]
        gii_values.append(gii)

    except:
        gii_values.append(np.nan)

# Add GII feature
df7["GII"] = gii_values

# ---------------------------------------------------
# Check NaN values
# ---------------------------------------------------

print("NaN values in composition_oxid:")
print(df7[oxid_col].isna().sum())

print("\nNaN values in GII:")
print(df7["GII"].isna().sum())

df7.head(3)

NaN values in composition_oxid:
0

NaN values in GII:
206


,material_id,formula,composition,structure,functional,band_gap,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,...,spacegroup_num,crystal_system,crystal_system_int,is_centrosymmetric,n_symmetry_ops,density,vpa,packing fraction,max packing efficiency,GII
0,mp-1183115,AcAlO3,"(Ac, Al, O)","[[0. 0. 0.] Ac, [1.92931693 1.92931693 1.92931...",GGA Static,4.1024,7.4,14.995948,1.8,12.2,...,221,cubic,1,True,48,8.728230,11.490283,0.730268,0.446920,0.120009
1,mp-1183052,AcBO3,"(Ac, B, O)","[[0. 0. 0.] Ac, [1.860834 1.860834 1.860834] B...",GGA Static,0.8071,5.8,11.761840,1.6,12.2,...,221,cubic,1,True,96,9.206879,10.309625,0.705091,0.446920,0.817748
2,mp-866101,AcCrO3,"(Ac, Cr, O)","[[0. 0. 0.] Ac, [1.97214345 1.97215113 1.97213...",GGA+U Static,2.0031,9.6,19.998860,2.0,10.8,...,221,cubic,1,True,48,8.848788,12.272569,0.737706,0.446918,0.139450


In [ ]:
# Save data_before_cleaning CSV
df7.to_csv("data_before_cleaning.csv", index=False)

print("CSV file saved as data_before_cleaning.csv")
print("Final shape:", df7.shape)

CSV file saved as data_before_cleaning.csv
Final shape: (2592, 306)


#Data Cleaning

In [ ]:
# Remove rows where GII column has NaN values
df8 = df7.dropna(subset=['GII']).reset_index(drop=True)
df8 = df8.dropna(subset=['gap_AO']).reset_index(drop=True)

print(df8.shape)

(2344, 306)


In [ ]:
# Keep first 5 columns unchanged
first_cols = df8.iloc[:, :5]

# Columns from 5th onward
remaining_cols = df8.iloc[:, 5:]

# Keep only numerical columns
numeric_cols = remaining_cols.select_dtypes(
    exclude=['object', 'bool']
)

# Identify removed columns
removed_cols = [
    col for col in remaining_cols.columns
    if col not in numeric_cols.columns
]

# Combine back
df8 = pd.concat([first_cols, numeric_cols], axis=1)

# Print removed columns
print("Columns removed:")
for col in removed_cols:
    print(col)

print(f"\nTotal columns removed: {len(removed_cols)}")

# Check remaining columns
print("\nRemaining columns and dtypes:")
print(df8.dtypes)

print("\nFinal shape:", df8.shape)

Columns removed:
composition_oxid
structure_oxi
compound possible
HOMO_character
HOMO_element
LUMO_character
LUMO_element
crystal_system
is_centrosymmetric

Total columns removed: 9

Remaining columns and dtypes:
material_id                object
formula                    object
composition                object
structure                  object
functional                 object
                           ...   
density                   float64
vpa                       float64
packing fraction          float64
max packing efficiency    float64
GII                       float64
Length: 297, dtype: object

Final shape: (2344, 297)


In [ ]:
#Remove "composition", "structure" column
df8 = df8.drop(columns=["composition", "structure"])
# Display all columns for first 2 rows
pd.set_option('display.max_columns', None)

display(df8.head(2))

,material_id,formula,functional,band_gap,avg_Atomic_Number,avg_Atomic_Weight,avg_Period,avg_group,avg_families,avg_Metal,avg_Nonmetal,avg_Metalliod,avg_Mendeleev_Number,avg_l_quantum_number,avg_Atomic_Radius,avg_Miracle_Radius_[pm],avg_Covalent_Radius,avg_Zunger_radii_sum,avg_ionic_radius,avg_crystal_radius,avg_Pauling_Electronegativity,avg_MB_electonegativity,avg_Gordy_electonegativity,avg_Mulliken_EN,avg_Allred-Rockow_electronegativity,avg_metallic_valence,avg_number_of_valence_electrons,avg_gilmor_number_of_valence_electron,avg_valence_s,avg_valence_p,avg_valence_d,avg_valence_f,avg_Number_of_unfilled_s_valence_electrons,avg_Number_of_unfilled_p_valence_electrons,avg_Number_of_unfilled_d_valence_electrons,avg_Number_of_unfilled_f_valence_electrons,avg_outer_shell_electrons,avg_1st_ionization_potential_(kJ/mol),avg_polarizability(A^3),avg_Melting_point_(K),avg_Boiling_Point_(K),avg_Density_(g/mL),avg_specific_heat_(J/g_K)_,avg_heat_of_fusion_(kJ/mol)_,avg_heat_of_vaporization_(kJ/mol)_,avg_thermal_conductivity_(W/(m_K))_,avg_heat_atomization(kJ/mol),avg_Cohesive_energy,dev_Atomic_Number,dev_Atomic_Weight,dev_Period,dev_group,dev_families,dev_Metal,dev_Nonmetal,dev_Metalliod,dev_Mendeleev_Number,dev_l_quantum_number,dev_Atomic_Radius,dev_Miracle_Radius_[pm],dev_Covalent_Radius,dev_Zunger_radii_sum,dev_ionic_radius,dev_crystal_radius,dev_Pauling_Electronegativity,dev_MB_electonegativity,dev_Gordy_electonegativity,dev_Mulliken_EN,dev_Allred-Rockow_electronegativity,dev_metallic_valence,dev_number_of_valence_electrons,dev_gilmor_number_of_valence_electron,dev_valence_s,dev_valence_p,dev_valence_d,dev_valence_f,dev_Number_of_unfilled_s_valence_electrons,dev_Number_of_unfilled_p_valence_electrons,dev_Number_of_unfilled_d_valence_electrons,dev_Number_of_unfilled_f_valence_electrons,dev_outer_shell_electrons,dev_1st_ionization_potential_(kJ/mol),dev_polarizability(A^3),dev_Melting_point_(K),dev_Boiling_Point_(K),dev_Density_(g/mL),dev_specific_heat_(J/g_K)_,dev_heat_of_fusion_(kJ/mol)_,dev_heat_of_vaporization_(kJ/mol)_,dev_thermal_conductivity_(W/(m_K))_,dev_heat_atomization(kJ/mol),dev_Cohesive_energy,range_Atomic_Number,range_Atomic_Weight,range_Period,range_group,range_families,range_Metal,range_Nonmetal,range_Metalliod,range_Mendeleev_Number,range_l_quantum_number,range_Atomic_Radius,range_Miracle_Radius_[pm],range_Covalent_Radius,range_Zunger_radii_sum,range_ionic_radius,range_crystal_radius,range_Pauling_Electronegativity,range_MB_electonegativity,range_Gordy_electonegativity,range_Mulliken_EN,range_Allred-Rockow_electronegativity,range_metallic_valence,range_number_of_valence_electrons,range_gilmor_number_of_valence_electron,range_valence_s,range_valence_p,range_valence_d,range_valence_f,range_Number_of_unfilled_s_valence_electrons,range_Number_of_unfilled_p_valence_electrons,range_Number_of_unfilled_d_valence_electrons,range_Number_of_unfilled_f_valence_electrons,range_outer_shell_electrons,range_1st_ionization_potential_(kJ/mol),range_polarizability(A^3),range_Melting_point_(K),range_Boiling_Point_(K),range_Density_(g/mL),range_specific_heat_(J/g_K)_,range_heat_of_fusion_(kJ/mol)_,range_heat_of_vaporization_(kJ/mol)_,range_thermal_conductivity_(W/(m_K))_,range_heat_atomization(kJ/mol),range_Cohesive_energy,max_Atomic_Number,max_Atomic_Weight,max_Period,max_group,max_families,max_Metal,max_Nonmetal,max_Metalliod,max_Mendeleev_Number,max_l_quantum_number,max_Atomic_Radius,max_Miracle_Radius_[pm],max_Covalent_Radius,max_Zunger_radii_sum,max_ionic_radius,max_crystal_radius,max_Pauling_Electronegativity,max_MB_electonegativity,max_Gordy_electonegativity,max_Mulliken_EN,max_Allred-Rockow_electronegativity,max_metallic_valence,max_number_of_valence_electrons,max_gilmor_number_of_valence_electron,max_valence_s,max_valence_p,max_valence_d,max_valence_f,max_Number_of_unfilled_s_valence_electrons,max_Number_of_unfilled_p_valence_electrons,max_Number_of_unfilled_d_valence_electrons,max_Number_of_unfilled_f_valence_electrons,max_outer_

In [ ]:
# SCRIPT 1: Split, filter ELEMENTAL features on TRAIN ONLY,
# apply same filtered columns to test, save train/test CSVs

from sklearn.model_selection import train_test_split
import json

RANDOM_STATE = 10
TEST_SIZE = 0.15
CORR_THRESHOLD = 0.90
TARGET_COL = "band_gap"

# ------------------------------------------------------------
# STEP 1: Split the full dataframe first
train_df, test_df = train_test_split(
    df8,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

# ------------------------------------------------------------
# STEP 2: Identify column groups

elemental_cols = list(df8.columns[4:-13])
geometrical_cols = list(df8.columns[-13:-8])
structural_cols = list(df8.columns[-8:])

# Make absolutely sure target is not included in any feature group
elemental_cols = [c for c in elemental_cols if c != TARGET_COL]
geometrical_cols = [c for c in geometrical_cols if c != TARGET_COL]
structural_cols = [c for c in structural_cols if c != TARGET_COL]

print(f"Elemental columns   : {len(elemental_cols)}")
print(f"Geometrical columns : {len(geometrical_cols)}")
print(f"Structural columns  : {len(structural_cols)}")


# ------------------------------------------------------------
# STEP 3: Correlation-based filtering on ELEMENTAL features
# using TRAINING split only (no dtype cleanup needed here --
# already handled upstream)

X_train_elemental = train_df[elemental_cols]

target_train = train_df[TARGET_COL]

zero_var_cols = list(X_train_elemental.columns[
    X_train_elemental.nunique() <= 1
])

X_train_elemental = X_train_elemental.drop(
    columns=zero_var_cols
)

target_corr = X_train_elemental.apply(
    lambda col: col.corr(target_train)
).abs()

target_corr = target_corr.dropna().sort_values(
    ascending=False
)

features_corr_train = X_train_elemental[
    target_corr.index
].corr(method="pearson").abs()

kept_elemental_features = []

for feature in target_corr.index:

    if not kept_elemental_features:
        kept_elemental_features.append(feature)
        continue

    if features_corr_train.loc[
        feature, kept_elemental_features
    ].max() <= CORR_THRESHOLD:
        kept_elemental_features.append(feature)

# Elemental columns removed by correlation filtering --
# everything that did NOT survive (either dropped as
# zero-variance, or dropped for exceeding the correlation
# threshold against a more target-relevant feature)
correlation_removed_features = [
    c for c in target_corr.index if c not in kept_elemental_features
]
all_removed_elemental_features = zero_var_cols + correlation_removed_features

n_before = len(elemental_cols)
n_after = len(kept_elemental_features)

print(f"\nElemental columns before filtering : {n_before}")
print(f"  Removed (zero-variance)          : {len(zero_var_cols)}")
print(f"  Removed (correlation > {CORR_THRESHOLD})     : {len(correlation_removed_features)}")
print(f"Elemental columns remaining        : {n_after}")


# ------------------------------------------------------------
# STEP 4: Build final feature list -- filtered elemental +
# untouched geometrical + untouched structural. Removed
# elemental columns are, by construction, excluded here.

final_columns = (
    kept_elemental_features
    + geometrical_cols
    + structural_cols
)

# Remove duplicates just in case
final_columns = list(dict.fromkeys(final_columns))

# Explicitly remove target and metadata columns
non_feature_cols = [
    "formula",
    "material_id",
    "functional",
    TARGET_COL
]

final_columns = [
    c for c in final_columns
    if c not in non_feature_cols
]

non_feature_cols_present = [
    c for c in non_feature_cols
    if c in df8.columns
]


# ------------------------------------------------------------
# STEP 4.1: Construct train and test dataframes -- correlated
# elemental columns are removed here simply by never being
# included in final_columns.

train_df_final = train_df[
    non_feature_cols_present + final_columns
].reset_index(drop=True)

test_df_final = test_df[
    non_feature_cols_present + final_columns
].reset_index(drop=True)

# Sanity check: confirm the removed columns are actually absent
still_present = [c for c in all_removed_elemental_features if c in train_df_final.columns]
if still_present:
    print(f"WARNING: {len(still_present)} supposedly-removed columns "
          f"are still present in train_df_final: {still_present}")
else:
    print(f"\nConfirmed: all {len(all_removed_elemental_features)} removed "
          f"elemental columns are absent from train_df_final and test_df_final.")


# ------------------------------------------------------------
# STEP 4.2: Explicitly drop any correlation-removed elemental
# columns from train_df_final / test_df_final.

train_df_final = train_df_final.drop(
    columns=correlation_removed_features, errors="ignore"
)
test_df_final = test_df_final.drop(
    columns=correlation_removed_features, errors="ignore"
)

still_present_after_explicit_drop = [
    c for c in correlation_removed_features if c in train_df_final.columns
]
if still_present_after_explicit_drop:
    print(f"WARNING: {len(still_present_after_explicit_drop)} correlation-removed "
          f"columns still present after explicit drop: {still_present_after_explicit_drop}")
else:
    print(f"\nConfirmed: all {len(correlation_removed_features)} correlation-removed "
          f"elemental columns are absent from train_df_final and test_df_final.")

# ------------------------------------------------------------
# DIAGNOSTIC: if columns are still persisting despite the drop
# above, this block identifies the most likely cause -- run
# this and inspect the printed output before saving.

dup_cols_df8 = df8.columns[df8.columns.duplicated()].tolist()
if dup_cols_df8:
    print(f"\nDIAGNOSTIC: df8 has DUPLICATE column names: {dup_cols_df8}. "
          f"This can cause .drop(columns=...) to behave unexpectedly, since "
          f"pandas treats duplicate-named columns as a single label mapping "
          f"to multiple columns.")

overlap_geom_struct = set(correlation_removed_features) & (set(geometrical_cols) | set(structural_cols))
if overlap_geom_struct:
    print(f"\nDIAGNOSTIC: {len(overlap_geom_struct)} correlation-removed elemental "
          f"column name(s) also appear in geometrical_cols/structural_cols: "
          f"{overlap_geom_struct}. If so, dropping by name removes ALL columns "
          f"sharing that name, including the geometrical/structural one you "
          f"want to keep -- these need to be renamed to be distinguishable "
          f"before this pipeline will work correctly.")

still_in_train_df = [c for c in correlation_removed_features if c in train_df.columns]
still_in_train_df_final_cols = [c for c in correlation_removed_features if c in train_df_final.columns]
print(f"\nDIAGNOSTIC: of {len(correlation_removed_features)} correlation-removed "
      f"columns -- present in train_df (original, pre-filter): {len(still_in_train_df)} "
      f"(expected, this is normal) -- present in train_df_final (post-filter): "
      f"{len(still_in_train_df_final_cols)} (should be 0).")

# ------------------------------------------------------------
# Check target column

print("\nTarget column check:")
print("Train band_gap columns:",
      (train_df_final.columns == TARGET_COL).sum())

print("Test band_gap columns :",
      (test_df_final.columns == TARGET_COL).sum())

print(f"\nFinal train_df shape: {train_df_final.shape}")
print(f"Final test_df shape : {test_df_final.shape}")


# ------------------------------------------------------------
# STEP 5: Save train/test CSVs, and a JSON recording exactly
# which final columns belong to which group, AND which
# elemental columns were removed and why.

final_elemental_cols = [c for c in final_columns if c in kept_elemental_features]
final_geometrical_cols = [c for c in final_columns if c in geometrical_cols]
final_structural_cols = [c for c in final_columns if c in structural_cols]

print("\nFinal column counts:")
print(f"Elemental   : {len(final_elemental_cols)}")
print(f"Geometrical : {len(final_geometrical_cols)}")
print(f"Structural  : {len(final_structural_cols)}")

# Sanity check: every final feature column should be accounted
# for in exactly one of the three groups
accounted_for = set(final_elemental_cols) | set(final_geometrical_cols) | set(final_structural_cols)
unaccounted = [c for c in final_columns if c not in accounted_for]
if unaccounted:
    print(f"WARNING: {len(unaccounted)} columns not assigned to any group: {unaccounted}")

# ------------------------------------------------------------
# STEP 5.0: Guaranteed final removal -- immediately before
# saving, re-intersect against whatever is ACTUALLY still
# present at this point, so nothing between here and the save
# can silently reintroduce a removed column.

final_removal_check = [c for c in correlation_removed_features if c in train_df_final.columns]
if final_removal_check:
    print(f"\nRemoving {len(final_removal_check)} correlation-removed column(s) "
          f"detected immediately before save: {final_removal_check}")
    train_df_final = train_df_final.drop(columns=final_removal_check, errors="ignore")
    test_df_final = test_df_final.drop(columns=final_removal_check, errors="ignore")

assert not any(c in train_df_final.columns for c in correlation_removed_features), \
    "Correlation-removed columns are still present in train_df_final -- see DIAGNOSTIC output above."
assert not any(c in test_df_final.columns for c in correlation_removed_features), \
    "Correlation-removed columns are still present in test_df_final -- see DIAGNOSTIC output above."

train_df_final.to_csv("train_df.csv", index=False)
test_df_final.to_csv("test_df.csv", index=False)

column_groups = {
    "elemental": final_elemental_cols,
    "geometrical": final_geometrical_cols,
    "structural": final_structural_cols,
    "removed_elemental_zero_variance": zero_var_cols,
    "removed_elemental_correlation": correlation_removed_features,
    "removed_elemental_all": all_removed_elemental_features,
}
with open("column_groups.json", "w") as f:
    json.dump(column_groups, f, indent=2)

print("\nSaved train_df.csv, test_df.csv, and column_groups.json")

Elemental columns   : 278
Geometrical columns : 5
Structural columns  : 8

Elemental columns before filtering : 278
  Removed (zero-variance)          : 6
  Removed (correlation > 0.9)     : 135
Elemental columns remaining        : 137

Confirmed: all 141 removed elemental columns are absent from train_df_final and test_df_final.

Confirmed: all 135 correlation-removed elemental columns are absent from train_df_final and test_df_final.

DIAGNOSTIC: of 135 correlation-removed columns -- present in train_df (original, pre-filter): 135 (expected, this is normal) -- present in train_df_final (post-filter): 0 (should be 0).

Target column check:
Train band_gap columns: 1
Test band_gap columns : 1

Final train_df shape: (1992, 154)
Final test_df shape : (352, 154)

Final column counts:
Elemental   : 137
Geometrical : 5
Structural  : 8

Saved train_df.csv, test_df.csv, and column_groups.json


#Data for Polymorphic analysis

In [ ]:
# Combine train_df and test_df
polymorph = pd.concat([test_df_final , train_df_final], axis=0, ignore_index=True)

# Save as CSV
polymorph.to_csv("polymorph.csv", index=False)

print("Polymorph dataframe shape:", polymorph.shape)
print("Saved as: polymorph.csv")

Polymorph dataframe shape: (2344, 154)
Saved as: polymorph.csv
